# <font color="#418FDE" size="6.5" uppercase>**Daten aufteilen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes. 
- Berücksichtigen Klassen, Gruppen und Zeitordnung bei manuellen Splits. 
- Erkennen und dokumentieren typische Formen von Datenleckage. 


## **1. Split Grundlagen**

### **1.1. Warum Daten aufteilen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_01.jpg?v=1784788177" width="250">



>* Aufteilung prüft Leistung auf neuen Fällen
>* Train, Validierung und Test trennen Aufgaben

>* Training lernt Muster, Validierung steuert Entscheidungen
>* Testdaten prüfen unabhängig wie zukünftige Fälle

>* Reproduzierbare Splits sichern vergleichbare Ergebnisse
>* Indizes dokumentieren verwendete Fälle nachvollziehbar



### **1.2. Indizes zufällig mischen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_02.jpg?v=1784788179" width="250">



>* Indizes mischen verhindert verzerrte Datensplits
>* Vollständige Zeilen bleiben dabei zusammen

>* Fester Zufallszustand macht Splits wiederholbar
>* Stabile Testdaten ermöglichen faire Modellvergleiche

>* Gemischte Indizes machen Splits nachvollziehbar
>* Gespeicherte Aufteilungen sichern faire Vergleiche



In [ ]:
#@title Python-Code - Indizes zufällig mischen

# Wir mischen Zeilenindizes reproduzierbar für faire Datensplits.
# Ein fester Zufallszustand erzeugt dieselbe Reihenfolge.
# Die Ausgabe zeigt stabile Trainings-, Validierungs- und Testindizes.

import numpy as np
import pandas as pd

# Ein kleiner sortierter Datensatz macht Verzerrungen sichtbar.
data = pd.DataFrame(
    {
        "customer_id": range(100, 112),
        "group": ["A", "A", "A", "A", "B", "B", "B", "B", "C", "C", "C", "C"],
        "score": [51, 53, 55, 57, 70, 72, 74, 76, 88, 90, 92, 94],
    }
)

# Die ursprünglichen Positionen bleiben als Indizes erhalten.
indices = np.arange(len(data))

# Ohne Mischen landen sortierte Gruppen leicht getrennt.
plain_train = indices[:8]
plain_test = indices[8:]

# Ein fester Generator mischt die Indizes reproduzierbar.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(indices)

# Die gemischten Indizes werden anschließend in drei Mengen geteilt.
train_indices = shuffled_indices[:7]
validation_indices = shuffled_indices[7:9]
test_indices = shuffled_indices[9:]

# Eine einfache Prüfung schützt vor verlorenen oder doppelten Zeilen.
all_split_indices = np.concatenate(
    [train_indices, validation_indices, test_indices]
)

if sorted(all_split_indices.tolist()) != indices.tolist():
    raise ValueError("Die Split-Indizes decken nicht alle Zeilen genau einmal ab.")

# Die Ausgabe bleibt kurz und zeigt den Kernunterschied.
print("Unsortierter Test ohne Mischen:", plain_test.tolist())
print("Gemischte Trainingsindizes:", train_indices.tolist())
print("Gemischte Validierungsindizes:", validation_indices.tolist())
print("Gemischte Testindizes:", test_indices.tolist())
print("Testgruppen ohne Mischen:", data.loc[plain_test, "group"].tolist())
print("Testgruppen mit Mischen:", data.loc[test_indices, "group"].tolist())
print("Reproduzierbar mit seed 42:", shuffled_indices[:5].tolist())



### **1.3. Splitgrößen prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_01_03.jpg?v=1784788181" width="250">



>* Tatsächliche Splitgrößen immer kritisch prüfen
>* Kleine Datensätze liefern oft instabile Bewertungen

>* Gerundete Splitgrößen exakt zusammenzählen
>* Keine Daten verlieren oder doppelt nutzen

>* Splitgrößen sichern Nachvollziehbarkeit und Dokumentation
>* Indizes ermöglichen spätere Wiederherstellung derselben Aufteilung



In [ ]:
#@title Python-Code - Splitgrößen prüfen

# Wir prüfen Splitgrößen mit gespeicherten Indizes.
# Ganze Zeilen erzeugen kleine Rundungsabweichungen.
# Die Ausgabe bestätigt vollständige, getrennte Teilmengen.

import numpy as np
import pandas as pd

# Ein kleiner Datensatz macht Rundungseffekte gut sichtbar.
row_count = 53
all_indices = np.arange(row_count)

# Ein fester Generator macht die Aufteilung reproduzierbar.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(all_indices)

# Gewünschte Anteile werden in ganze Zeilenzahlen umgerechnet.
train_size = int(row_count * 0.70)
validation_size = int(row_count * 0.15)
test_size = row_count - train_size - validation_size

# Die gespeicherten Indizes definieren die drei Teilmengen eindeutig.
train_indices = shuffled_indices[:train_size]
validation_indices = shuffled_indices[train_size:train_size + validation_size]
test_indices = shuffled_indices[train_size + validation_size:]

# Diese Prüfung erkennt verlorene oder doppelte Beobachtungen.
combined_indices = np.concatenate(
    [train_indices, validation_indices, test_indices]
)
unique_count = len(np.unique(combined_indices))

# Eine kleine Tabelle zeigt Sollanteile und tatsächliche Größen.
summary = pd.DataFrame(
    {
        "Split": ["Training", "Validierung", "Test"],
        "Zeilen": [len(train_indices), len(validation_indices), len(test_indices)],
        "Anteil": [len(train_indices) / row_count, len(validation_indices) / row_count, len(test_indices) / row_count],
    }
)

# Gerundete Prozentwerte sind für Anfänger leichter lesbar.
summary["Anteil"] = (summary["Anteil"] * 100).round(1)

# Die Ausgabe bleibt kurz und konzentriert auf die Splitprüfung.
print("Splitgrößen bei 53 Beobachtungen:")
print(summary.to_string(index=False))
print(f"Summe der Splitgrößen: {len(combined_indices)} von {row_count}")
print(f"Eindeutige Indizes: {unique_count} von {row_count}")
print(f"Überschneidungsfrei und vollständig: {unique_count == row_count}")



## **2. Spezielle Splits**

### **2.1. Klassen ausgewogen aufteilen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_01.jpg?v=1784788173" width="250">



>* Klassenanteile in allen Splits ähnlich halten
>* Seltene Klassen für aussagekräftige Bewertung sichern

>* Seltene Klassen brauchen besondere Aufmerksamkeit
>* Klassenbewusste Splits machen Bewertungen stabiler

>* Klassenanteile prüfen und sorgfältig dokumentieren
>* Daten sauber trennen, Leckage vermeiden



In [ ]:
#@title Python-Code - Klassen ausgewogen aufteilen

# Wir vergleichen zufällige und klassenbewusste Aufteilungen.
# Stratifikation erhält Klassenanteile in allen Teilmengen.
# Die Tabelle zeigt stabilere Anteile pro Split.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Ein kleiner Datensatz simuliert eine seltene positive Klasse.
rng = np.random.default_rng(42)
labels = np.array([0] * 90 + [1] * 10)
indices = np.arange(len(labels))

# Zuerst teilen wir ohne Beachtung der Klassen.
random_train, random_temp = train_test_split(
    indices, test_size=0.4, random_state=42, shuffle=True
)
random_valid, random_test = train_test_split(
    random_temp, test_size=0.5, random_state=42, shuffle=True
)

# Danach teilen wir mit Stratifikation nach der Zielklasse.
strat_train, strat_temp = train_test_split(
    indices, test_size=0.4, random_state=42, stratify=labels
)
strat_valid, strat_test = train_test_split(
    strat_temp, test_size=0.5, random_state=42, stratify=labels[strat_temp]
)

# Diese Funktion berechnet Anzahl und Anteil der seltenen Klasse.
def positive_summary(split_indices):
    split_labels = labels[split_indices]
    positive_count = int(split_labels.sum())
    positive_share = round(positive_count / len(split_labels), 2)
    return positive_count, positive_share

# Wir sammeln die wichtigsten Kennzahlen übersichtlich ein.
rows = []
for method, train_idx, valid_idx, test_idx in [
    ("zufällig", random_train, random_valid, random_test),
    ("stratifiziert", strat_train, strat_valid, strat_test),
]:
    for split_name, split_idx in [
        ("Train", train_idx), ("Validierung", valid_idx), ("Test", test_idx)
    ]:
        count, share = positive_summary(split_idx)
        rows.append([method, split_name, len(split_idx), count, share])

# Die Ausgabe zeigt, warum Klassenanteile dokumentiert werden sollten.
summary = pd.DataFrame(
    rows,
    columns=["Methode", "Teilmenge", "Größe", "Klasse 1", "Anteil 1"],
)

print("Gesamtdatensatz: 100 Zeilen, Klasse 1 Anteil: 0.10")
print(summary.to_string(index=False))



### **2.2. Gruppen getrennt halten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_02.jpg?v=1784788175" width="250">



>* Gruppen enthalten abhängige, zusammengehörige Beobachtungen
>* Gruppen nie über Splits verteilen

>* Evaluation muss neue Gruppen realistisch nachbilden
>* Ganze Konten oder Personen getrennt halten

>* Gruppeneinheit und Generalisierungsziel zuerst klären
>* Verteilungen prüfen und Split nachvollziehbar dokumentieren



In [ ]:
#@title Python-Code - Gruppen getrennt halten

# Dieses Beispiel zeigt gruppenbasierte Datenaufteilung.
# Gruppen dürfen nicht zwischen Teilmengen wandern.
# Die Ausgabe vergleicht zufällige und gruppierte Splits.

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split

# Wir erzeugen kleine Messdaten mit mehreren Zeilen pro Person.
rng = np.random.default_rng(42)
groups = np.repeat(np.arange(1, 9), 3)
measurements = rng.normal(loc=groups * 2, scale=0.4)

# Die Zielklasse hängt hier absichtlich stark von der Gruppe ab.
target = (groups >= 5).astype(int)
data = pd.DataFrame({"group": groups, "measurement": measurements, "target": target})

# Eine einfache Zufallsaufteilung ignoriert die Gruppenzugehörigkeit.
row_indices = np.arange(len(data))
train_rows, test_rows = train_test_split(row_indices, test_size=0.33, random_state=42)

# GroupShuffleSplit hält jede Gruppe vollständig zusammen.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
group_split = splitter.split(data, data["target"], groups=data["group"])
group_train_rows, group_test_rows = next(group_split)

# Diese Hilfsfunktion zählt Gruppen, die in beiden Teilmengen vorkommen.
def shared_group_count(train_index, test_index):
    train_groups = set(data.loc[train_index, "group"])
    test_groups = set(data.loc[test_index, "group"])
    return len(train_groups.intersection(test_groups))

# Eine kurze Prüfung macht die wichtigste Annahme sichtbar.
if len(data) != len(groups):
    raise ValueError("Die Gruppenzuordnung passt nicht zu den Daten.")

random_leakage = shared_group_count(train_rows, test_rows)
group_leakage = shared_group_count(group_train_rows, group_test_rows)

print("scikit-learn wird für GroupShuffleSplit verwendet.")
print(f"Zeilen im Datensatz: {len(data)}")
print(f"Zufälliger Split: geteilte Gruppen = {random_leakage}")
print(f"Gruppen-Split: geteilte Gruppen = {group_leakage}")
print("Merksatz: Bei neuen Personen müssen ganze Personen getrennt bleiben.")



### **2.3. Zeitordnung beachten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_02_03.jpg?v=1784788171" width="250">



>* Zeitliche Reihenfolge beim Split bewahren
>* Zukunftsdaten vermeiden, Verteilungsänderungen realistisch prüfen

>* Nach klaren Zeitfenstern aufteilen
>* Nur damals verfügbare Merkmale verwenden

>* Mehrere Zeitfenster zeigen Modellrobustheit.
>* Zeitlogik klar dokumentieren und begründen.



In [ ]:
#@title Python-Code - Zeitordnung beachten

# Dieses Beispiel zeigt zeitgerechte Datensplits.
# Zufällige Splits können Zukunftsinformationen mischen.
# Die Ausgabe vergleicht beide Aufteilungen.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wir erzeugen kleine Bestelldaten mit klarer Zeitreihenfolge.
rng = np.random.default_rng(42)
months = pd.date_range("2021-01-01", periods=36, freq="MS")
trend = np.linspace(80, 150, len(months))
season = 12 * np.sin(np.linspace(0, 6 * np.pi, len(months)))
orders = np.round(trend + season + rng.normal(0, 4, len(months))).astype(int)

# Die Tabelle enthält nur Informationen pro Monat.
data = pd.DataFrame({"month": months, "orders": orders})
if len(data) != 36:
    raise ValueError("Die Beispieldaten sollten 36 Monate enthalten.")

# Ein zeitgerechter Split nutzt frühere Monate zuerst.
train_end = pd.Timestamp("2022-12-01")
valid_end = pd.Timestamp("2023-06-01")
train_mask = data["month"] <= train_end
valid_mask = (data["month"] > train_end) & (data["month"] <= valid_end)
test_mask = data["month"] > valid_end

# Ein zufälliger Split ignoriert die zeitliche Reihenfolge.
shuffled_indices = rng.permutation(data.index.to_numpy())
random_train = shuffled_indices[:24]
random_valid = shuffled_indices[24:30]
random_test = shuffled_indices[30:]

# Wir fassen die Zeiträume beider Strategien knapp zusammen.
print("Zeitgerechter Split: Train 2021-01 bis 2022-12, Valid 2023-01 bis 2023-06, Test 2023-07 bis 2023-12")
print("Zufälliger Split: Train enthält Monate von " + str(data.loc[random_train, "month"].min().date()) + " bis " + str(data.loc[random_train, "month"].max().date()))
print("Zufälliger Split: Test enthält Monate von " + str(data.loc[random_test, "month"].min().date()) + " bis " + str(data.loc[random_test, "month"].max().date()))
print("Leckage-Hinweis: Zufälliges Training kann spätere Monate als der Test enthalten.")

# Die Grafik markiert die zeitgerechten Bereiche chronologisch.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(data["month"], data["orders"], marker="o", color="black", label="Bestellungen")
ax.axvspan(data.loc[train_mask, "month"].min(), train_end, color="tab:blue", alpha=0.15, label="Training")
ax.axvspan(pd.Timestamp("2023-01-01"), valid_end, color="tab:orange", alpha=0.18, label="Validierung")
ax.axvspan(pd.Timestamp("2023-07-01"), data["month"].max(), color="tab:green", alpha=0.15, label="Test")
ax.set_title("Zeitordnung beim manuellen Split beachten")
ax.set_xlabel("Monat")
ax.set_ylabel("Bestellungen")
ax.legend()
plt.show()



## **3. Datenleckage erkennen**

### **3.1. Zielwertleckage erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_01.jpg?v=1784788185" width="250">



>* Zielwertleckage verrät die spätere Antwort.
>* Prüfe Merkmale zum Prognosezeitpunkt fachlich.

>* Leckage versteckt sich oft in abgeleiteten Merkmalen
>* Prüfe Verfügbarkeit jedes Merkmals zeitlich genau

>* Merkmalsentscheidungen nachvollziehbar dokumentieren
>* Modellgüte mit Prozesswissen prüfen



In [ ]:
#@title Python-Code - Zielwertleckage erkennen

# Dieses Beispiel zeigt Zielwertleckage in Trainingsdaten.
# Ein verdächtiges Merkmal verrät den Zielwert fast direkt.
# Die Ausgabe vergleicht faire und undichte Modellgüte.

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Wir erzeugen kleine Kundendaten mit bekanntem Zielwert.
rng = np.random.default_rng(42)
row_count = 400
income = rng.normal(3000, 700, row_count)
late_payments = rng.poisson(1.2, row_count)

# Der Zielwert entsteht aus plausiblen, vorher bekannten Merkmalen.
risk_score = -0.002 * income + 0.9 * late_payments + rng.normal(0, 1, row_count)
defaulted = (risk_score > -4.2).astype(int)

# Dieses Merkmal wäre erst nach dem Ausfall bekannt.
collection_status = np.where(defaulted == 1, "Inkasso", "Normal")
leak_flag = (collection_status == "Inkasso").astype(int)

# Wir prüfen eine einfache Annahme zur Datenform.
data = pd.DataFrame(
    {"income": income, "late_payments": late_payments, "leak_flag": leak_flag}
)
if len(data) != len(defaulted):
    raise ValueError("Merkmale und Zielwert haben unterschiedlich viele Zeilen.")

# Beide Splits nutzen dieselben Zeilen für einen fairen Vergleich.
train_index, test_index = train_test_split(
    np.arange(row_count), test_size=0.3, stratify=defaulted, random_state=42
)

# Das faire Modell nutzt nur Informationen vom Prognosezeitpunkt.
fair_features = ["income", "late_payments"]
leaky_features = ["income", "late_payments", "leak_flag"]

# Wir trainieren zwei einfache Modelle mit und ohne Leckage.
fair_model = LogisticRegression(max_iter=200, random_state=42)
fair_model.fit(data.loc[train_index, fair_features], defaulted[train_index])
leaky_model = LogisticRegression(max_iter=200, random_state=42)
leaky_model.fit(data.loc[train_index, leaky_features], defaulted[train_index])

# Die Testgenauigkeit zeigt den verdächtigen Leistungssprung.
fair_pred = fair_model.predict(data.loc[test_index, fair_features])
leaky_pred = leaky_model.predict(data.loc[test_index, leaky_features])
fair_accuracy = accuracy_score(defaulted[test_index], fair_pred)
leaky_accuracy = accuracy_score(defaulted[test_index], leaky_pred)

# Eine perfekte Korrelation ist ein starkes Warnsignal.
leak_match_rate = np.mean(leak_flag == defaulted)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Faire Testgenauigkeit: {fair_accuracy:.3f}")
print(f"Undichte Testgenauigkeit: {leaky_accuracy:.3f}")
print(f"Trefferquote des verdächtigen Merkmals: {leak_match_rate:.3f}")
print("Dokumentation: leak_flag entfernen, weil es erst nach dem Zielereignis entsteht.")



### **3.2. Skalierung ohne Leckage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_02.jpg?v=1784788183" width="250">



>* Skalierung kann unbemerkt Datenleckage erzeugen
>* Testverteilungen verfälschen die Leistungsbewertung

>* Skalierungswerte nur aus Trainingsdaten berechnen
>* Validierungs- und Testdaten unverändert transformieren

>* Transformationen nur aus Trainingssicht planen
>* Skalierungsschritte dokumentieren und auf Leckage prüfen



In [ ]:
#@title Python-Code - Skalierung ohne Leckage

# Dieses Beispiel zeigt Skalierung ohne Datenleckage.
# Trainingsdaten bestimmen die erlaubten Skalierungsregeln.
# Die Ausgabe vergleicht falsche und saubere Parameter.

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Kleine Daten mit absichtlich verschobenen Testwerten.
feature = np.array([10, 11, 12, 13, 14, 15, 16, 17, 80, 90], dtype=float)
target = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
X = feature.reshape(-1, 1)

# Die letzten zwei Werte simulieren unbekannte spätere Testdaten.
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2, shuffle=False
)

# Falsch: Der Skalierer lernt aus allen Daten.
leaky_scaler = StandardScaler()
leaky_scaler.fit(X)
leaky_test_value = leaky_scaler.transform(X_test)[0, 0]

# Richtig: Der Skalierer lernt nur aus Trainingsdaten.
clean_scaler = StandardScaler()
clean_scaler.fit(X_train)
clean_test_value = clean_scaler.transform(X_test)[0, 0]

# Eine einfache Prüfung macht die Annahme sichtbar.
if X_train.shape[0] + X_test.shape[0] != X.shape[0]:
    raise ValueError("Die Aufteilung passt nicht zur Datenmenge.")

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsmittelwert sauber: {clean_scaler.mean_[0]:.2f}")
print(f"Mittelwert mit Leckage: {leaky_scaler.mean_[0]:.2f}")
print(f"Erster Testwert roh: {X_test[0, 0]:.2f}")
print(f"Skaliert sauber: {clean_test_value:.2f}")
print(f"Skaliert mit Leckage: {leaky_test_value:.2f}")
print("Merksatz: fit nur auf Training, transform auf alle Teile.")



### **3.3. Split Protokoll**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_06/Lecture_A/image_03_03.jpg?v=1784788186" width="250">



>* Split-Entscheidungen nachvollziehbar dokumentieren
>* Zusammengehörige Einheiten gemeinsam aufteilen

>* Zeit, Gruppen und seltene Klassen dokumentieren
>* Split passend zur Anwendung begründen

>* Verarbeitungsschritte nur trainingsbasiert festlegen
>* Indizes und Zufallszustände reproduzierbar dokumentieren



In [ ]:
#@title Python-Code - Split Protokoll

# Dieses Beispiel erstellt ein kleines Split Protokoll.
# Es zeigt Datenleckage durch geteilte Gruppen.
# Die Ausgabe vergleicht riskante und saubere Splits.

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split

# Wir bauen kleine Beispieldaten mit wiederholten Kundengruppen.
rng = np.random.default_rng(42)
groups = np.repeat(np.arange(1, 13), 3)
labels = np.where(groups <= 4, "hoch", "niedrig")
row_ids = np.arange(len(groups))

# Diese Tabelle steht für die dokumentierte Datenversion.
data = pd.DataFrame(
    {"row_id": row_ids, "customer_id": groups, "risk_class": labels}
)

# Ein normaler Zufallssplit kann dieselben Gruppen verteilen.
train_idx, test_idx = train_test_split(
    data.index, test_size=0.33, random_state=42, stratify=data["risk_class"]
)

# Ein Gruppensplit hält alle Zeilen einer Kundengruppe zusammen.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
group_train_idx, group_test_idx = next(
    splitter.split(data, data["risk_class"], groups=data["customer_id"])
)

# Diese Funktion zählt Gruppen, die in beiden Teilen vorkommen.
def count_leaking_groups(train_indices, test_indices):
    train_groups = set(data.loc[train_indices, "customer_id"])
    test_groups = set(data.loc[test_indices, "customer_id"])
    return len(train_groups.intersection(test_groups))

# Das Protokoll hält Entscheidungen und Leckageprüfungen fest.
protocol = pd.DataFrame(
    {
        "split": ["zufaellig", "gruppenbasiert"],
        "train_rows": [len(train_idx), len(group_train_idx)],
        "test_rows": [len(test_idx), len(group_test_idx)],
        "leaking_groups": [
            count_leaking_groups(train_idx, test_idx),
            count_leaking_groups(group_train_idx, group_test_idx),
        ],
    }
)

# Die Ausgabe ist ein kompaktes Split Protokoll.
print("Split Protokoll: gleiche Kundengruppe in Training und Test?")
print(protocol.to_string(index=False))
print("Merksatz: leaking_groups sollte bei gruppierten Daten null sein.")



# <font color="#418FDE" size="6.5" uppercase>**Daten aufteilen**</font>


In this lecture, you learned to:
- Erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen mit Indizes. 
- Berücksichtigen Klassen, Gruppen und Zeitordnung bei manuellen Splits. 
- Erkennen und dokumentieren typische Formen von Datenleckage. 

In the next Lecture (Lecture B), we will go over 'Verluste und Metriken'